In [21]:
import pandas as pd
from dataclasses import dataclass

import numpy as np
from pandas import Series, DataFrame



import mw_analysis.utils as utils
import mw_analysis.snirf as snirf

CHANNELS = [
    "acc_x", "acc_y", "acc_z",
    "ang_x", "ang_y", "ang_z",
    "temp",
    "ir_l", "red_l", "amb_l",
    "ir_r", "red_r", "amb_r",
    "ir_p", "red_p", "amb_p",
    "battery_voltage", "timestamp"
]

MAPPINGS = {
    "S1_D1": ("red_l", "ir_l"),
    "S1_D2": ("red_r", "ir_r"),
    "S1_D3": ("red_p", "ir_p"),
}


def clean_stream(df):
    dataframe = df.copy()

    dataframe['ir_l'] = dataframe['ir_l'] - dataframe['amb_l']
    dataframe['red_l'] = dataframe['red_l'] - dataframe['amb_l']
    dataframe['ir_r'] = dataframe['ir_r'] - dataframe['amb_r']
    dataframe['red_r'] = dataframe['red_r'] - dataframe['amb_r']
    dataframe['ir_p'] = dataframe['ir_p'] - dataframe['amb_p']
    dataframe['red_p'] = dataframe['red_p'] - dataframe['amb_p']

    return dataframe


# Processed datasets
datasets = []

for subject_id in range(1, 9):
    stream_df, events_df = utils.mw_h5_to_df(f"../data/participant{subject_id}-mendi.h5", CHANNELS)

    # Clean the events df
    events_df = utils.clean_events(events_df)
    stream_df = clean_stream(stream_df)

    # convert to optical density
    stream_df = utils.od(stream_df, events_df)

    # Convert to HbO/Hb
    hbo_df = stream_df[["timestamp"]]

    for name, (ch1, ch2) in MAPPINGS.items():
        HbO, Hb = utils.mbll(stream_df[ch1].values, stream_df[ch2].values)
        hbo_df[f"{name} hbo"] = utils.iir_filter(HbO)
        hbo_df[f"{name} hb"] = utils.iir_filter(Hb)

    # Add to the datasets HAAIIIL TO THE KIIING
    dataset = snirf.NirscordDataset(subject_id, hbo_df, events_df)

    datasets.append(dataset)

    # Write to SNIRF file
    snirf.to_snirf(dataset, f"../data/processed/participant{subject_id}-mendi.snirf")




In [22]:
import plotly.graph_objects as go


colours = ["red", "orange", "yellow", "green", "blue", "purple", "brown", "pink"]

for dataset in datasets:
    fig = go.Figure()

    #
    time = np.arange(dataset.stream_df.iloc[-1]["timestamp"])

    # --- left channel
    fig_left = go.Figure()

    for i, name in enumerate(MAPPINGS.keys()):
        fig_left.add_trace(go.Scatter(
            x=time, y=dataset.stream_df[f"{name} hbo"],
            mode='lines', name=f'{name} Δ[HbO]',
            line=dict(color=colours[i]),
            hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
        ))

    fig_left.show()

In [23]:
import itertools

error_epochs = []
no_error_epochs = []

def extract_epochs(stream_df, events_df):
    epochs = []

    for index, row in events_df.iterrows():
        epochs.append(stream_df[
            (stream_df["timestamp"] >= row.timestamp - 30) &
            (stream_df["timestamp"] < row.timestamp + 10)
        ].reset_index(drop=True))

    return epochs

for dataset in datasets:
    error_epochs.append(extract_epochs(dataset.stream_df, dataset.events_df[dataset.events_df["value"] == 0]))
    no_error_epochs.append(extract_epochs(dataset.stream_df, dataset.events_df[dataset.events_df["value"] == 1]))

error_epochs = list(itertools.chain(*error_epochs))
no_error_epochs = list(itertools.chain(*no_error_epochs))


In [24]:
import pandas as pd

# --- time axis in seconds
time = np.arange(-30, 10)



# ---


average_sart_errors = pd.concat(error_epochs, axis=0).groupby(level=0).mean().dropna()
average_sart_no_errors = pd.concat(no_error_epochs, axis=0).groupby(level=0).mean().dropna()

for i, name in enumerate(MAPPINGS.keys()):
    fig_left = go.Figure()
    fig_left.add_trace(go.Scatter(
        x=time, y=average_sart_errors[f"{name} hbo"],
        mode='lines', name='SART Error',
        line=dict(color='blue'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig_left.add_trace(go.Scatter(
        x=time, y=average_sart_no_errors[f"{name} hbo"],
        mode='lines', name='SART No Error',
        line=dict(color='green'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig_left.update_layout(
        title=f'Average SART {name} HbO concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )
    fig_left.show()

